# U-Net land-cover training (Colab)

Trains the PyTorch U-Net land-cover classifier on a GPU. Run this from **VS Code**:
`Select Kernel` -> `Colab` -> sign in with your Google account -> `New Colab Server`
-> pick **GPU** (the free T4 is enough) -> connect.

Local machines never train this model -- only run inference on the weights this
notebook produces. After this notebook finishes, on your own machine run:

```bash
python scripts/pull_models.py --model unet
python scripts/run_landcover_unet_inference.py
```

**One-time setup**: none needed in advance -- the auth cell below prompts a masked
paste-in box for this repo's service-account JSON key
(`credentials/nus-iss-urban-heat-sg-*.json`) each session, via `getpass` (a plain
Jupyter stdin prompt). Never an interactive Google login, matching this project's
service-account-only convention everywhere else. (`google.colab.files.upload()`
and Colab Secrets were tried first but don't reliably work through VS Code's
remote-kernel connection to Colab -- confirmed 2026-08-05.)

In [13]:
# --- Repo sync -----------------------------------------------------------
# Colab's /content disk is empty every session -- clone (or pull, if this
# session already has it) the public repo so `src/`/`config/` are importable
# the same "sys.path manipulation, no packaging" way every local script uses.
import os
import subprocess
import sys

REPO_URL = "https://github.com/EngineerKX/urban-heat-cooling-priority.git"
REPO_DIR = "/content/urban-heat-cooling-priority"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo ready at", REPO_DIR)

Repo ready at /content/urban-heat-cooling-priority


In [14]:
# --- Install trimmed training-only dependencies ---------------------------
# requirements-colab.txt deliberately excludes torch (Colab's pre-installed,
# CUDA-matched build) and tensorflow (unused everywhere in this project now).
%pip install -q -r {REPO_DIR}/requirements-colab.txt

In [15]:
# --- Auth: service account, not interactive Google login -------------------
# Paste the full contents of your service-account JSON key when prompted
# below (open credentials/nus-iss-urban-heat-sg-*.json in a text editor,
# copy everything, Ctrl+V into the box that appears). Uses getpass -- a
# plain stdin prompt, standard Jupyter protocol -- not a browser widget,
# since google.colab.files.upload() hung indefinitely through VS Code's
# remote-kernel connection to Colab (confirmed 2026-08-05). getpass also
# never echoes the input back into the cell's saved output, unlike a plain
# input() -- this repo is public on GitHub, so that distinction matters.
import getpass
import os

key_json = getpass.getpass("Paste the full contents of your service-account JSON key, then press Enter: ")

key_path = "/content/gee_key.json"
with open(key_path, "w") as f:
    f.write(key_json)

os.environ["GEE_PRIVATE_KEY_PATH"] = key_path
os.environ["GEE_SERVICE_ACCOUNT"] = "urban-heat-pipeline@nus-iss-urban-heat-sg.iam.gserviceaccount.com"
os.environ["GEE_PROJECT_ID"] = "nus-iss-urban-heat-sg"
os.environ["GEE_EXPORT_BUCKET"] = "nus-iss-urban-heat-sg-exports"
print("Service-account credentials staged.")

Service-account credentials staged.


In [16]:
import ee
import mlflow
import pandas as pd
import torch

from config import settings
from config.settings import (
    DRY_SEASON_MONTHS,
    GCS_MODEL_BUCKET,
    S2_CLOUD_PROB_MAX,
    S2_UTM_CRS,
    SG_BBOX,
    TARGET_SCALE_M,
    UNET_MODEL_SAVE_PATH,
    VALIDATION_SAMPLE_GCS_PREFIX,
    YEARS,
)
from src.ingest.gee import init_ee
from src.ingest.subzones import as_ee_feature_collection, dissolve_boundary, fetch_subzones_geojson
from src.ingest.worldcover import get_worldcover_bucket_image
from src.landcover.rf_baseline import build_feature_image, build_training_region
from src.landcover.unet_data import (
    build_training_stack,
    export_training_patches,
    parse_training_patches,
    training_fingerprint,
)
from src.landcover.unet_train import train_unet
from src.utils import gcs
from src.utils.experiment_tracking import export_run_summary, start_run
from src.utils.seed import set_all_seeds

In [17]:
set_all_seeds()
init_ee()
print("GPU available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

EE initialized OK (service account), project: nus-iss-urban-heat-sg
GPU available: True - Tesla T4


In [18]:
# --- Pull the hand-labeled validation sample -------------------------------
# Gitignored (data/interim/**), so a fresh git clone doesn't bring it along.
# It can't be regenerated in Colab either -- labeling requires a human and
# the Streamlit app. Pushed once (and after every relabel) via:
#   python -c "from src.utils import gcs; from config.settings import GCS_MODEL_BUCKET, VALIDATION_SAMPLE_GCS_PREFIX; gcs.upload_file('data/interim/validation_sample/validation_sample_200_labeled.csv', GCS_MODEL_BUCKET, f'{VALIDATION_SAMPLE_GCS_PREFIX}.csv')"
from pathlib import Path

VALIDATION_CSV = Path("data/interim/validation_sample/validation_sample_200_labeled.csv")
VALIDATION_CSV.parent.mkdir(parents=True, exist_ok=True)
gcs.download_blob(GCS_MODEL_BUCKET, f"{VALIDATION_SAMPLE_GCS_PREFIX}.csv", VALIDATION_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)
print(f"Loaded {len(validation_df)} validation points.")

Loaded 200 validation points.


In [19]:
# --- Build the GEE feature stack -------------------------------------------
# Identical to scripts/train_landcover_unet.py's (deleted) top half -- same
# feature bands/season window/exclusion buffer as the RF baseline, kept
# deliberately identical so the RF-vs-U-Net comparison isolates model choice.
sg_bbox = ee.Geometry.Rectangle(list(SG_BBOX))
subzones_fc = as_ee_feature_collection(fetch_subzones_geojson())
boundary = dissolve_boundary(subzones_fc)

feature_image, valid_mask = build_feature_image(sg_bbox, boundary, YEARS, DRY_SEASON_MONTHS, S2_CLOUD_PROB_MAX)
wc_bucket_image = get_worldcover_bucket_image(boundary, S2_UTM_CRS, TARGET_SCALE_M, valid_mask=valid_mask)

training_region, _ = build_training_region(boundary, validation_df)
training_stack = build_training_stack(feature_image, wc_bucket_image, training_region)
print("Feature stack built.")

No cache found — fetching URA subzones from data.gov.sg (one-time).
Sanitized 2 property name(s) containing '.': ['SHAPE.AREA -> SHAPE_AREA', 'SHAPE.LEN -> SHAPE_LEN']
Dissolved Singapore boundary built. Approx area: 788.3 km²
(Sanity check: Singapore's land area is ~730-735 km².)
Singapore boundary area: 788.30 km²
Training region area (post-exclusion): 788.16 km²
Excluded around validation points: 138,444 m²
✅ Validation points are spatially excluded from the training region.
Feature stack built.


In [20]:
# --- Export (or reuse the GCS-cached) training patches ---------------------
# export_training_patches checks gs://<bucket>/unet_train_patches/<fingerprint-
# of-validation-csv>/ first -- only the very first run for a given labeled
# sample actually pays for the multi-minute-plus GEE export; every later
# session (yours, your partner's, a hyperparameter-only retrain) just
# downloads the cached shards. Pass force_export=True to bypass deliberately.
train_patch_dir = export_training_patches(training_stack, training_region, VALIDATION_CSV)
train_loader, val_loader, n_patches = parse_training_patches(train_patch_dir)

Patch export started: gs://nus-iss-urban-heat-sg-exports/unet_train_patches/319f7b912b47d3060522e2c81b76fd8a4c0a1937e41a3fa0d973aef6bb964a0f/unet_train
  ...READY
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...

In [21]:
# --- Train -------------------------------------------------------------
# force_retrain=False (the default): if this cell is re-run within the same
# Colab session after an earlier successful run, it reuses the already-
# trained model sitting in this session's /content disk instead of retraining
# from scratch -- useful when re-running cells while debugging downstream code.
data_fingerprint = training_fingerprint(VALIDATION_CSV)

with start_run("unet"):
    mlflow.log_params({
        "unet_patch_size": settings.UNET_PATCH_SIZE,
        "unet_batch_size": settings.UNET_BATCH_SIZE,
        "unet_epochs": settings.UNET_EPOCHS,
        "unet_learning_rate": settings.UNET_LEARNING_RATE,
        "unet_base_filters": settings.UNET_BASE_FILTERS,
        "unet_early_stop_patience": settings.UNET_EARLY_STOP_PATIENCE,
        "unet_train_val_split": settings.UNET_TRAIN_VAL_SPLIT,
        "n_validation_points": len(validation_df),
        "n_train_patches": n_patches,
    })

    model, history = train_unet(train_loader, val_loader, data_fingerprint=data_fingerprint, force_retrain=False)

    if history is not None:
        for epoch, (loss, val_loss, val_metric) in enumerate(
            zip(history["train_loss"], history["val_loss"], history["val_metric"])
        ):
            mlflow.log_metrics({"train_loss": loss, "val_loss": val_loss, "val_accuracy": val_metric}, step=epoch)
        mlflow.log_metric("best_val_loss", min(history["val_loss"]))

print(f"Model saved to {UNET_MODEL_SAVE_PATH}")

2026/08/05 13:14:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/05 13:14:15 INFO mlflow.store.db.utils: Updating database tables
2026/08/05 13:14:17 INFO mlflow.tracking.fluent: Experiment with name 'landcover_classifiers' does not exist. Creating a new experiment.


UNet(
  (backbone): UNetBackbone(
    (enc1): Sequential(
      (0): Conv2d(9, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU(inplace=True)
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU(inplace=True)
    )
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (enc2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU(inplace=True)
    )
    (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (enc3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU(inplace=True)
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU(inplace=True)
    )
    (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=

In [22]:
# --- Push the trained weights + this run's MLflow summary to GCS ----------
# Colab's local mlflow.db is ephemeral -- export_run_summary + upload_text
# is what lets `python scripts/pull_models.py` re-log this run into your
# local shared MLflow store afterward (see src/utils/experiment_tracking.py).
import json
import time

if history is not None:
    summary = export_run_summary(
        "unet", "landcover_classifiers",
        params={
            "unet_patch_size": settings.UNET_PATCH_SIZE,
            "unet_epochs": settings.UNET_EPOCHS,
            "unet_learning_rate": settings.UNET_LEARNING_RATE,
            "n_train_patches": n_patches,
        },
        metrics_history={
            "train_loss": history["train_loss"],
            "val_loss": history["val_loss"],
            "val_accuracy": history["val_metric"],
        },
    )
    gcs.upload_text(json.dumps(summary), GCS_MODEL_BUCKET, f"training_runs/unet_{int(time.time())}.json")
    print("Run summary pushed for local MLflow import.")

!python {REPO_DIR}/scripts/push_models.py --model unet

Run summary pushed for local MLflow import.
[unet] Uploading /content/urban-heat-cooling-priority/models/unet_landcover.pt -> gs://nus-iss-urban-heat-sg-exports/models/unet_landcover.pt ...
[unet] Pushed OK.


## Next steps (on your own machine, no GPU needed)

```bash
python scripts/pull_models.py --model unet
python scripts/run_landcover_unet_inference.py
```

The first command downloads the trained weights (verified via sha256) and
imports this run into your local MLflow store. The second runs full-Singapore
CPU inference + tile reconstruction -- U-Net inference was never the GPU
bottleneck, only the 30-epoch training loop above was.